# Session 32: Logistic Regression Classification

This notebook implements Logistic Regression for at-risk classification.
- 1 = at-risk (G3 < 10)
- 0 = successful (G3 >= 10)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Load data

In [ ]:
data_dir = Path("data/processed")
X_full = pd.read_parquet(data_dir / "X_full.parquet")
y = pd.read_parquet(data_dir / "y_full.parquet")
y = y["G3"] if "G3" in y.columns else y.iloc[:, 0]
print(f"Features: {X_full.shape}, Target: {len(y)}")

## 3. Train/test split and classification labels

In [ ]:
Xtr_f, Xte_f, ytr, yte = train_test_split(X_full, y, test_size=0.20, random_state=42)
yc = (y < 10).astype(int)
yctr = yc.loc[ytr.index]
ycte = yc.loc[yte.index]
print("Training:", yctr.value_counts().to_dict())
print("Test:", ycte.value_counts().to_dict())

## 4. Fit Logistic Regression

In [ ]:
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
clf.fit(Xtr_f, yctr)
y_pred = clf.predict(Xte_f)
y_proba = clf.predict_proba(Xte_f)[:, 1]

print("Accuracy:", accuracy_score(ycte, y_pred))
print("Precision:", precision_score(ycte, y_pred))
print("Recall:", recall_score(ycte, y_pred))
print("F1:", f1_score(ycte, y_pred))
print("ROC_AUC:", roc_auc_score(ycte, y_proba))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(ycte, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Successful", "At-risk"])
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.show()

## 6. Classification Report

In [ ]:
print(classification_report(ycte, y_pred, target_names=["Successful", "At-risk"]))

## 7. Coefficients

In [ ]:
coef_df = pd.DataFrame({
    "Feature": Xtr_f.columns,
    "Coefficient": clf.named_steps["logisticregression"].coef_[0]
})
coef_df["Abs"] = coef_df["Coefficient"].abs()
display(coef_df.sort_values("Abs", ascending=False).head(10))

## 8. Classification Row

In [ ]:
tn, fp, fn, tp = cm.ravel()
classification_row = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(ycte, y_pred),
    "Precision": precision_score(ycte, y_pred),
    "Recall": recall_score(ycte, y_pred),
    "F1": f1_score(ycte, y_pred),
    "ROC_AUC": roc_auc_score(ycte, y_proba),
    "True_Negative": tn,
    "False_Positive": fp,
    "False_Negative": fn,
    "True_Positive": tp
}])
display(classification_row)

## Session 32 Complete

✅ Logistic Regression with StandardScaler
✅ Classification metrics computed
✅ Confusion matrix displayed
✅ Classification row created

## Session 34: Decision Tree Classifier

This section adds a Decision Tree classifier with different depths.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# Verify prerequisites
for obj in ["Xtr_f", "Xte_f", "yctr", "ycte"]:
    if obj not in globals():
        raise NameError(f"Missing required object: {obj}")

print("Session 34 prerequisites verified")
print("Training shape:", Xtr_f.shape)
print("Test shape:", Xte_f.shape)


In [ ]:
def evaluate_classifier(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
    }
print("Evaluator ready")


In [ ]:
# Train Decision Tree with different depths
depths = [3, 5, 7, 10, None]
results = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(Xtr_f, yctr)
    predictions = dt.predict(Xte_f)
    probabilities = dt.predict_proba(Xte_f)[:, 1]
    metrics = evaluate_classifier(ycte, predictions, probabilities)
    results.append({
        "Model": f"DT_depth_{depth if depth else "None"}",
        "Max_Depth": depth,
        **metrics
    })
    print(f"DT depth {depth if depth else "None"} completed - F1: {metrics["f1"]:.4f}")

results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
print("\nDecision Tree Results:")
print(results_df.to_string())


In [ ]:
# Save results
from pathlib import Path
repo_root = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists()), Path.cwd())
output_dir = repo_root / "reports" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(output_dir / "session34_decision_tree_results.csv", index=False)
print("Results saved to:", output_dir / "session34_decision_tree_results.csv")


### Session 34 Interpretation

- **Decision Trees** split data based on feature values
- Different depths test the trade-off between interpretability and performance
- Shallower trees are more interpretable but may underfit
- Deeper trees capture complex patterns but risk overfitting


## Session 33: KNN, SVM, and Naive Bayes Classification

This section adds three classifiers: K-Nearest Neighbors, Support Vector Machine, and Gaussian Naive Bayes.
All classifiers use the same training and test data with scaling for KNN and SVM.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Verify prerequisites
for obj in ["Xtr_f", "Xte_f", "yctr", "ycte"]:
    if obj not in globals():
        raise NameError(f"Missing required object: {obj}")

print("Session 33 prerequisites verified")
print("Training shape:", Xtr_f.shape)
print("Test shape:", Xte_f.shape)


In [ ]:
def evaluate_classifier(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
    }
print("Evaluator ready")


In [ ]:
# Train KNN, SVM, and Gaussian Naive Bayes
results = []
for code, name, family, clf in [
    ("KNN", "K-Nearest Neighbors", "Instance-based", KNeighborsClassifier()),
    ("SVM", "Support Vector Machine", "Maximum-margin", SVC(probability=True, random_state=42)),
    ("NB", "Gaussian Naive Bayes", "Probabilistic", GaussianNB()),
]:
    pipeline = make_pipeline(StandardScaler(), clf)
    pipeline.fit(Xtr_f, yctr)
    predictions = pipeline.predict(Xte_f)
    probabilities = pipeline.predict_proba(Xte_f)[:, 1]
    metrics = evaluate_classifier(ycte, predictions, probabilities)
    results.append({
        "Model": code,
        "Full_Model_Name": name,
        "Model_Family": family,
        "Scaling_Used": True,
        **metrics
    })
    print(f"{code} completed - F1: {metrics["f1"]:.4f}")

results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
results_df.insert(0, "Session33_F1_Rank", range(1, len(results_df) + 1))

print("\nSession 33 Classification Results:")
print(results_df.to_string())


In [ ]:
# Save results to CSV
from pathlib import Path
repo_root = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists()), Path.cwd())
output_dir = repo_root / "reports" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(output_dir / "session33_classification_rows.csv", index=False)
print("Results saved to:", output_dir / "session33_classification_rows.csv")


### Session 33 Interpretation

- **KNN** classifies based on similarity in scaled feature space
- **SVM** finds a maximum-margin decision boundary
- **Gaussian Naive Bayes** assumes conditional independence of predictors

Naive Bayes provides a useful baseline even with the independence assumption.


## Session 35: Random Forest Classification

This section trains a 300-tree Random Forest classifier and compares it with the single Decision Tree on the same held-out test set.

- Class `0` = at-risk student
- Class `1` = successful student
- Primary safety-oriented metric = at-risk recall (`pos_label=0`)
- Complementary discrimination metric = ROC AUC

The test set is used only for final evaluation, not for model selection or tuning.

In [ ]:
# Session 35 - imports, prerequisite validation, and Random Forest training
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

required_objects = ["Xtr_f", "Xte_f", "yctr", "ycte"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(
        "Run the earlier classification data-preparation cells first. "
        f"Missing objects: {missing_objects}"
    )

# Preserve a Session 35-specific reference to the held-out labels.
ycte_s35 = ycte

rfc = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)
rfc.fit(Xtr_f, yctr)

forest_predictions = rfc.predict(Xte_f)
forest_success_probability = rfc.predict_proba(Xte_f)[:, 1]
forest_at_risk_probability = rfc.predict_proba(Xte_f)[:, 0]

print("SESSION 35 RANDOM FOREST TRAINING PASSED")


In [ ]:
# Session 35 - model evaluation, Decision Tree comparison, confusion matrix, and ROC curve
if "eval_clf" in globals() and callable(eval_clf):
    print("Existing eval_clf output:")
    print(eval_clf(ycte_s35, forest_predictions, forest_success_probability))

rf_accuracy = accuracy_score(ycte_s35, forest_predictions)
rf_success_precision = precision_score(ycte_s35, forest_predictions, pos_label=1, zero_division=0)
rf_success_recall = recall_score(ycte_s35, forest_predictions, pos_label=1, zero_division=0)
rf_success_f1 = f1_score(ycte_s35, forest_predictions, pos_label=1, zero_division=0)
rf_roc_auc = roc_auc_score(ycte_s35, forest_success_probability)
rf_at_risk_precision = precision_score(ycte_s35, forest_predictions, pos_label=0, zero_division=0)
rf_at_risk_recall = recall_score(ycte_s35, forest_predictions, pos_label=0, zero_division=0)
rf_at_risk_f1 = f1_score(ycte_s35, forest_predictions, pos_label=0, zero_division=0)
rf_at_risk_auc = roc_auc_score(
    (np.asarray(ycte_s35) == 0).astype(int),
    forest_at_risk_probability,
)

tree_at_risk_recall = None
if "tree_classifier_row" in globals():
    for key in ("Recall_At_Risk", "At-Risk Recall", "At_Risk_Recall"):
        if key in tree_classifier_row:
            tree_at_risk_recall = float(tree_classifier_row[key])
            break
if tree_at_risk_recall is None:
    for prediction_name in ("tree_predictions", "dt_predictions", "dtc_predictions", "y_pred_tree"):
        if prediction_name in globals():
            tree_at_risk_recall = recall_score(
                ycte_s35,
                globals()[prediction_name],
                pos_label=0,
                zero_division=0,
            )
            break
if tree_at_risk_recall is None:
    raise NameError(
        "A Decision Tree result is required for the recall comparison. "
        "Run the Session 34 Decision Tree cells first."
    )

recall_change = rf_at_risk_recall - tree_at_risk_recall
recall_comparison = pd.DataFrame(
    {
        "Model": ["Decision Tree", "Random Forest"],
        "Recall_At_Risk": [tree_at_risk_recall, rf_at_risk_recall],
    }
)
recall_comparison["Difference_From_Tree"] = [0.0, recall_change]

print("Random Forest Classification Metrics")
print("------------------------------------")
print(f"Accuracy:          {rf_accuracy:.4f}")
print(f"At-risk precision: {rf_at_risk_precision:.4f}")
print(f"At-risk recall:    {rf_at_risk_recall:.4f}")
print(f"At-risk F1:        {rf_at_risk_f1:.4f}")
print(f"ROC AUC:           {rf_roc_auc:.4f}")
print(f"Tree recall:       {tree_at_risk_recall:.4f}")
print(f"Recall change:     {recall_change:+.4f}")
display(recall_comparison)

session35_figure_dir = Path("results/session35")
session35_figure_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    ycte_s35,
    forest_predictions,
    display_labels=["At risk", "Successful"],
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Session 35 Random Forest Confusion Matrix")
fig.tight_layout()
fig.savefig(session35_figure_dir / "session35_random_forest_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
RocCurveDisplay.from_predictions(
    (np.asarray(ycte_s35) == 0).astype(int),
    forest_at_risk_probability,
    name="Random Forest (at risk)",
    ax=ax,
)
ax.plot([0, 1], [0, 1], "--", color="gray", label="Chance")
ax.set_title("Session 35 At-Risk ROC Curve")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(session35_figure_dir / "session35_random_forest_at_risk_roc.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Session 35 - add exactly one Random Forest row to the classification table
random_forest_row = {
    "Model": "Random Forest",
    "Accuracy": rf_accuracy,
    "Precision_Success": rf_success_precision,
    "Recall_Success": rf_success_recall,
    "F1_Success": rf_success_f1,
    "ROC_AUC": rf_roc_auc,
    "Precision_At_Risk": rf_at_risk_precision,
    "Recall_At_Risk": rf_at_risk_recall,
    "F1_At_Risk": rf_at_risk_f1,
    "ROC_AUC_At_Risk": rf_at_risk_auc,
}
random_forest_row_df = pd.DataFrame([random_forest_row])

if "classification_table" not in globals() or not isinstance(classification_table, pd.DataFrame):
    classification_table = pd.DataFrame(columns=random_forest_row_df.columns)

classification_table = classification_table.copy()
for column in random_forest_row_df.columns:
    if column not in classification_table.columns:
        classification_table[column] = np.nan
for column in classification_table.columns:
    if column not in random_forest_row_df.columns:
        random_forest_row_df[column] = np.nan

random_forest_row_df = random_forest_row_df[classification_table.columns]
classification_table = classification_table[
    classification_table["Model"].astype(str).str.strip().str.lower().ne("random forest")
].copy()
classification_table = pd.concat(
    [classification_table, random_forest_row_df],
    ignore_index=True,
)

random_forest_rows = classification_table[
    classification_table["Model"].astype(str).str.strip().str.lower().eq("random forest")
]
assert len(random_forest_rows) == 1, "The table must contain exactly one Random Forest row."

display(classification_table)
print("RANDOM FOREST CLASSIFICATION ROW PASSED")


In [ ]:
# Session 35 - save and verify reproducible output artifacts
SESSION35_OUTPUT_DIR = Path("results/session35")
SESSION35_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classification_table_path = SESSION35_OUTPUT_DIR / "session35_classification_table.csv"
random_forest_row_path = SESSION35_OUTPUT_DIR / "session35_random_forest_row.csv"
recall_comparison_path = SESSION35_OUTPUT_DIR / "session35_recall_comparison.csv"

classification_table.to_csv(classification_table_path, index=False)
random_forest_row_df.to_csv(random_forest_row_path, index=False)
recall_comparison.to_csv(recall_comparison_path, index=False)

required_columns = [
    "Model", "Accuracy", "Precision_Success", "Recall_Success", "F1_Success",
    "ROC_AUC", "Precision_At_Risk", "Recall_At_Risk", "F1_At_Risk",
    "ROC_AUC_At_Risk",
]
missing_columns = [column for column in required_columns if column not in random_forest_row_df.columns]
assert not missing_columns, f"Missing Random Forest columns: {missing_columns}"
assert len(random_forest_row_df) == 1
assert random_forest_row_df.iloc[0]["Model"] == "Random Forest"

metric_values = random_forest_row_df[required_columns[1:]].to_numpy(dtype=float)
assert np.isfinite(metric_values).all()
assert ((metric_values >= 0) & (metric_values <= 1)).all()
assert classification_table_path.exists()
assert random_forest_row_path.exists()
assert recall_comparison_path.exists()

print("SESSION 35 OUTPUT ARTIFACT PASSED")
print("Saved:", classification_table_path)
print("Saved:", random_forest_row_path)
print("Saved:", recall_comparison_path)


### Session 35 Reflection: Why use ROC AUC with accuracy?

ROC AUC complements accuracy because accuracy reports correctness at one selected classification cutoff and can appear high when successful students greatly outnumber at-risk students. A model could predict the majority class for nearly everyone and still attain high accuracy while missing students who may need support. ROC AUC instead measures how well the model ranks or separates the two classes across all possible thresholds. It should therefore be interpreted together with at-risk recall and precision: recall shows how many truly at-risk students are detected, while precision indicates how many alerts correspond to genuinely at-risk students. These measures support intervention planning; they do not justify surveillance, labeling, or causal claims.